# Modern LLMs with Google Gemini (Free API)

In this notebook, we explore **modern large language models** using **Google Gemini**, which offers a **free tier** perfect for students. We'll learn practical skills for working with LLMs in production, including prompt engineering, few-shot learning, and systematic evaluation.

## Why Gemini for This Course?

✅ **Free API Access**: Generous free tier (60 requests/minute, 1,500/day)

✅ **No Credit Card Required**: Just need a Google account

✅ **Production-Ready**: Used by Google products

✅ **Competitive Quality**: Comparable to GPT-3.5 Turbo

✅ **Multimodal**: Can handle text, images, and more

## Learning Objectives

1. Set up and use Google Gemini API
2. Master prompt engineering techniques
3. Implement few-shot learning
4. Evaluate LLMs systematically
5. Understand cost vs performance trade-offs
6. Compare when to use fine-tuned BERT vs API-based LLMs

---

## 1. Getting Your Free Gemini API Key

### Step-by-Step Setup:

1. Go to [Google AI Studio](https://makersuite.google.com/app/apikey)
2. Sign in with your Google account
3. Click **"Get API Key"** → **"Create API key"**
4. Copy your API key

**Free Tier Limits:**
- 60 requests per minute
- 1,500 requests per day
- 1 million tokens per month
- Perfect for learning and course projects!

## 2. Setup & Installation

In [ ]:
# Install required packages
!pip install -q google-generativeai pandas numpy matplotlib seaborn

In [ ]:
import os
import json
import time
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import google.generativeai as genai

print("✅ Packages imported successfully!")

In [ ]:
# Configure API key
# Option 1: Direct (for learning)
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your key

# Option 2: Environment variable (more secure)
# GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)

print("✅ Gemini API configured!")
print("\n⚠️ Security Tip: Never commit API keys to version control!")
print("   Use environment variables or .env files in production.")

## 3. The Modern LLM Landscape

### Major Models & Providers

| Model | Provider | Cost | Key Features | Best For |
|-------|----------|------|--------------|----------|
| **Gemini Pro** | Google | **FREE** | Multimodal, 32K context | **Students, prototypes** |
| **GPT-3.5** | OpenAI | Paid | Fast, reliable | General purpose |
| **GPT-4** | OpenAI | Expensive | Best quality | Critical tasks |
| **Claude 3.5** | Anthropic | Paid | Long context (200K) | Document analysis |
| **LLaMA 2** | Meta | Free* | Open weights | Self-hosting |
| **Mistral** | Mistral AI | Free/Paid | Efficient | Edge deployment |

*Free to download, but you need compute resources to run

### When to Use What?

**Use Gemini (Free) when:**
- ✅ Learning and experimentation
- ✅ Course projects and assignments
- ✅ Prototyping new ideas
- ✅ Low-medium volume applications
- ✅ Budget constraints

**Use Paid APIs (GPT-4, Claude) when:**
- Production apps with revenue
- Need absolute best quality
- Critical business applications
- Can justify the cost

**Use Fine-tuned BERT when:**
- Have labeled training data
- Need <10ms latency
- Fixed task (classification, NER)
- Very high volume (>100K queries/day)

## 4. Working with Gemini API

In [ ]:
# List available models
print("Available Gemini Models:\n")
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        print(f"  • {model.name}")
        print(f"    Input limit: {model.input_token_limit:,} tokens")
        print(f"    Output limit: {model.output_token_limit:,} tokens")
        print()

### 4.1 Basic Text Generation

In [ ]:
def call_gemini(prompt: str, temperature: float = 0.7, max_tokens: int = 150) -> Dict:
    """
    Call Gemini API with error handling and metrics.
    
    Args:
        prompt: The user prompt
        temperature: Sampling temperature (0-1)
        max_tokens: Maximum tokens to generate
    
    Returns:
        Dictionary with response and metrics
    """
    model = genai.GenerativeModel('gemini-pro')
    
    # Configure generation
    generation_config = genai.types.GenerationConfig(
        temperature=temperature,
        max_output_tokens=max_tokens
    )
    
    start_time = time.time()
    
    try:
        response = model.generate_content(
            prompt,
            generation_config=generation_config
        )
        
        latency = time.time() - start_time
        
        # Approximate token counts (Gemini doesn't expose exact counts in free tier)
        input_tokens = len(prompt.split()) * 1.3  # Rough estimate
        output_tokens = len(response.text.split()) * 1.3
        
        return {
            'response': response.text,
            'model': 'gemini-pro',
            'input_tokens': int(input_tokens),
            'output_tokens': int(output_tokens),
            'total_tokens': int(input_tokens + output_tokens),
            'latency': latency,
            'cost': 0.0  # Free!
        }
    
    except Exception as e:
        return {
            'response': f"Error: {str(e)}",
            'model': 'gemini-pro',
            'error': str(e)
        }

print("✅ call_gemini() function defined")

In [ ]:
# Test basic completion
prompt = "Explain what a transformer is in machine learning in 2 sentences."
result = call_gemini(prompt)

print(f"Prompt: {prompt}\n")
print(f"Response: {result['response']}\n")
print(f"Model: {result['model']}")
print(f"Tokens: ~{result['input_tokens']} in + ~{result['output_tokens']} out")
print(f"Latency: {result['latency']:.3f}s")
print(f"Cost: ${result['cost']:.2f} (FREE!)")

---
## 5. Prompt Engineering

The quality of LLM outputs depends heavily on **how you ask**.

### Key Principles:

1. **Be Specific**: Clear instructions → better results
2. **Provide Context**: Give background information
3. **Show Examples**: Few-shot learning improves accuracy
4. **Specify Format**: Tell the model how to structure output
5. **Iterate**: Refine prompts based on results

### 5.1 Comparing Prompt Quality

In [ ]:
# Compare bad vs good prompts
prompts_comparison = [
    {
        'name': '❌ Bad: Vague',
        'prompt': 'Tell me about AI'
    },
    {
        'name': '✅ Better: Specific',
        'prompt': 'Explain what artificial intelligence is and list 3 main subfields, in 3 sentences.'
    },
    {
        'name': '✅✅ Best: Specific + Format',
        'prompt': '''Explain artificial intelligence.

Format your answer as:
1. Definition (1 sentence)
2. Three main subfields (bullet points)
3. One real-world application (1 sentence)'''
    }
]

print("Comparing Prompt Quality:\n")
print("="*80)

for item in prompts_comparison:
    print(f"\n{item['name']}:")
    print(f"Prompt: {item['prompt'][:60]}..." if len(item['prompt']) > 60 else f"Prompt: {item['prompt']}")
    result = call_gemini(item['prompt'], temperature=0.7, max_tokens=200)
    print(f"\nResponse:\n{result['response']}")
    print("\n" + "="*80)
    time.sleep(1)  # Rate limiting

### 5.2 Few-Shot Learning

Provide examples in the prompt to teach the model the task pattern.

In [ ]:
# Zero-shot vs Few-shot comparison

# Zero-shot
zero_shot_prompt = """Classify the sentiment of this text as positive, negative, or neutral.

Text: "This product is amazing!"
Sentiment:"""

# Few-shot
few_shot_prompt = """Classify the sentiment of each text as positive, negative, or neutral.

Text: "I love this product! It exceeded my expectations."
Sentiment: positive

Text: "Terrible quality. Waste of money."
Sentiment: negative

Text: "It's okay, nothing special."
Sentiment: neutral

Text: "This product is amazing!"
Sentiment:"""

print("Zero-shot vs Few-shot Learning:\n")
print("="*80)

print("\nZero-shot:")
result_zero = call_gemini(zero_shot_prompt, temperature=0.0, max_tokens=10)
print(f"Response: {result_zero['response']}")

time.sleep(1)  # Rate limiting

print("\nFew-shot:")
result_few = call_gemini(few_shot_prompt, temperature=0.0, max_tokens=10)
print(f"Response: {result_few['response']}")

print("\n" + "="*80)
print("\n💡 Few-shot typically gives more consistent, accurate results")

### 5.3 Chain-of-Thought Prompting

Ask the model to "think step by step" for better reasoning.

In [ ]:
# Without CoT
direct_prompt = """Q: A store has 15 apples. They sell 7 in the morning and receive 20 more in the afternoon. 
Then they sell 12 more. How many apples do they have?

A:"""

# With Chain-of-Thought
cot_prompt = """Q: A store has 15 apples. They sell 7 in the morning and receive 20 more in the afternoon. 
Then they sell 12 more. How many apples do they have?

A: Let's solve this step by step:"""

print("Chain-of-Thought Prompting:\n")
print("="*80)

print("\nDirect:")
result_direct = call_gemini(direct_prompt, temperature=0.0, max_tokens=100)
print(result_direct['response'])

time.sleep(1)

print("\n" + "="*80)
print("\nChain-of-Thought:")
result_cot = call_gemini(cot_prompt, temperature=0.0, max_tokens=150)
print(result_cot['response'])

print("\n" + "="*80)
print("\n💡 CoT helps LLMs show their reasoning and catch errors")

---
## 6. LLM Evaluation Framework

How do we systematically evaluate LLMs?

### 6.1 Task-Specific Evaluation

For tasks with ground truth, we can measure accuracy.

In [ ]:
# Create evaluation dataset
eval_dataset = [
    {
        'text': 'I absolutely loved this movie! Best film of the year.',
        'true_sentiment': 'positive'
    },
    {
        'text': 'Terrible service. Never coming back.',
        'true_sentiment': 'negative'
    },
    {
        'text': 'The product works as expected.',
        'true_sentiment': 'neutral'
    },
    {
        'text': 'Waste of time and money. Very disappointed.',
        'true_sentiment': 'negative'
    },
    {
        'text': 'Excellent quality! Highly recommend.',
        'true_sentiment': 'positive'
    }
]

def evaluate_sentiment_model(dataset: List[Dict]) -> Dict:
    """
    Evaluate Gemini on sentiment classification task.
    """
    predictions = []
    total_latency = 0
    
    for item in dataset:
        prompt = f"""Classify sentiment as: positive, negative, or neutral.

Text: "{item['text']}"
Sentiment:"""
        
        result = call_gemini(prompt, temperature=0.0, max_tokens=10)
        pred = result['response'].strip().lower()
        
        # Extract sentiment label
        if 'positive' in pred:
            pred_label = 'positive'
        elif 'negative' in pred:
            pred_label = 'negative'
        else:
            pred_label = 'neutral'
        
        predictions.append(pred_label)
        total_latency += result['latency']
        
        time.sleep(1)  # Rate limiting
    
    # Calculate accuracy
    true_labels = [item['true_sentiment'] for item in dataset]
    accuracy = sum([p == t for p, t in zip(predictions, true_labels)]) / len(dataset)
    
    return {
        'model': 'gemini-pro',
        'accuracy': accuracy,
        'total_cost': 0.0,  # Free!
        'avg_latency': total_latency / len(dataset),
        'predictions': predictions,
        'true_labels': true_labels
    }

# Evaluate
print("Running evaluation (this may take ~10 seconds due to rate limiting)...\n")
results = evaluate_sentiment_model(eval_dataset)

print("Sentiment Classification Evaluation:\n")
print(f"Model: {results['model']}")
print(f"Accuracy: {results['accuracy']:.2%}")
print(f"Total Cost: ${results['total_cost']:.2f} (FREE!)")
print(f"Avg Latency: {results['avg_latency']:.3f}s")
print("\nPredictions vs True Labels:")
for i, (pred, true) in enumerate(zip(results['predictions'], results['true_labels'])):
    match = "✓" if pred == true else "✗"
    print(f"  {i+1}. {match} Predicted: {pred:8} | True: {true}")

### 6.2 Model Comparison

Let's compare different approaches for sentiment classification.

In [ ]:
# Comparison data (simulated for paid APIs, actual for Gemini)
comparison_data = [
    {
        'Model': 'Fine-tuned BERT',
        'Accuracy': 0.94,
        'Avg Latency (ms)': 15,
        'Cost per 1K queries': '$0.10',
        'Setup Cost': 'High (training)',
        'Update Cost': 'High (retrain)'
    },
    {
        'Model': 'Gemini (zero-shot)',
        'Accuracy': results['accuracy'],
        'Avg Latency (ms)': results['avg_latency'] * 1000,
        'Cost per 1K queries': '$0.00',
        'Setup Cost': 'None',
        'Update Cost': 'Low (edit prompt)'
    },
    {
        'Model': 'GPT-3.5 (zero-shot)',
        'Accuracy': 0.85,
        'Avg Latency (ms)': 500,
        'Cost per 1K queries': '$0.50',
        'Setup Cost': 'Low (prompt)',
        'Update Cost': 'Low (edit prompt)'
    },
    {
        'Model': 'GPT-4 (zero-shot)',
        'Accuracy': 0.96,
        'Avg Latency (ms)': 800,
        'Cost per 1K queries': '$5.00',
        'Setup Cost': 'Low (prompt)',
        'Update Cost': 'Low (edit prompt)'
    }
]

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("MODEL COMPARISON: Sentiment Classification Task")
print("="*100)
print(df_comparison.to_string(index=False))
print("="*100)

In [ ]:
# Visualize trade-offs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convert cost strings to floats
costs = [float(c.replace('$', '')) for c in df_comparison['Cost per 1K queries']]

# Plot 1: Accuracy vs Cost
colors = ['#4285F4', '#34A853', '#FBBC04', '#EA4335']  # Google colors
axes[0].scatter(costs, df_comparison['Accuracy'], s=200, alpha=0.6, c=colors)

for i, row in df_comparison.iterrows():
    axes[0].annotate(row['Model'], 
                     (costs[i], row['Accuracy']),
                     fontsize=9, ha='right', xytext=(-5, 0),
                     textcoords='offset points')

axes[0].set_xlabel('Cost per 1K Queries ($)', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy vs Cost Trade-off', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].set_xscale('symlog')  # Handle $0 cost

# Highlight Gemini's sweet spot
gemini_idx = df_comparison[df_comparison['Model'] == 'Gemini (zero-shot)'].index[0]
axes[0].scatter([costs[gemini_idx]], [df_comparison.iloc[gemini_idx]['Accuracy']], 
                s=300, facecolors='none', edgecolors='green', linewidths=3,
                label='Best for Students')
axes[0].legend()

# Plot 2: Latency comparison
axes[1].barh(df_comparison['Model'], df_comparison['Avg Latency (ms)'], 
             color=colors, edgecolor='black')
axes[1].set_xlabel('Average Latency (ms)', fontsize=12)
axes[1].set_title('Inference Latency Comparison', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Key Insights:")
print("  • Gemini: FREE, good accuracy, perfect for learning!")
print("  • Fine-tuned BERT: Best latency, low cost, but requires training")
print("  • GPT-4: Best accuracy but 1000x more expensive than Gemini")
print("  • For students: Gemini is the clear winner!")

---
## 7. Cost-Performance Analysis

Understanding the economics of LLMs.

In [ ]:
# Cost projection at different scales
def calculate_monthly_cost(queries_per_day: int, cost_per_query: float) -> float:
    return queries_per_day * 30 * cost_per_query

query_volumes = [100, 1000, 10000, 100000]
cost_scenarios = []

for volume in query_volumes:
    cost_scenarios.append({
        'Queries/Day': f"{volume:,}",
        'Gemini (FREE)': f"$0.00",
        'GPT-3.5': f"${calculate_monthly_cost(volume, 0.0005):.2f}",
        'GPT-4': f"${calculate_monthly_cost(volume, 0.005):.2f}",
        'BERT (compute)': f"${calculate_monthly_cost(volume, 0.0001):.2f}"
    })

df_cost = pd.DataFrame(cost_scenarios)

print("\n💰 Monthly Cost Projections\n")
print(df_cost.to_string(index=False))
print("\n🎓 For Students:")
print("   • Gemini FREE tier: 1,500 requests/day = 45,000/month")
print("   • More than enough for all course projects!")
print("   • GPT-4 at 1K queries/day = $150/month - not feasible for students")
print("\n💡 Tip: Start with Gemini, upgrade to paid APIs only when absolutely needed")

---
## 8. Decision Framework: When to Use What?

A practical guide for choosing the right approach.

In [ ]:
decision_tree = """
┌─ Are you a student or learning?
│
├─ YES → Use Gemini (FREE)
│  └─ Perfect for: coursework, projects, experimentation
│
└─ NO → Production Application
   |
   ├─ Do you have labeled training data (>1000 examples)?
   │  |
   │  ├─ YES → Consider Fine-tuning
   │  │  ├─ Need very low latency (<50ms)? → Fine-tune BERT
   │  │  ├─ Need generation capability? → Fine-tune GPT
   │  │  └─ Requirements change often? → Use API instead
   │  │
   │  └─ NO → Use API-based LLMs
   │     ├─ Low-medium volume (<50K/day)? → Gemini Pro (paid tier)
   │     ├─ Need best quality? → GPT-4 or Claude 3.5
   │     ├─ Need long context (>100K)? → Claude 3.5 (200K)
   │     └─ Privacy-sensitive? → Self-host LLaMA/Mistral
   │
   └─ Special Considerations:
      • High volume (>100K/day)? → Fine-tuning becomes cost-effective
      • Rapidly evolving domain? → RAG + API for flexibility
      • Need citations? → RAG is essential
      • Offline/embedded? → Must fine-tune or use small models
"""

print(decision_tree)

---
## 9. Best Practices

### 9.1 Rate Limiting

In [ ]:
import time
from datetime import datetime, timedelta

class RateLimiter:
    """Simple rate limiter for Gemini API."""
    
    def __init__(self, max_per_minute=60):
        self.max_per_minute = max_per_minute
        self.requests = []
    
    def wait_if_needed(self):
        """Wait if we're at the rate limit."""
        now = datetime.now()
        # Remove requests older than 1 minute
        self.requests = [r for r in self.requests if now - r < timedelta(minutes=1)]
        
        if len(self.requests) >= self.max_per_minute:
            # Wait until the oldest request is >1 minute old
            sleep_time = 61 - (now - self.requests[0]).seconds
            if sleep_time > 0:
                print(f"⏳ Rate limit reached, waiting {sleep_time}s...")
                time.sleep(sleep_time)
        
        self.requests.append(now)

# Usage
limiter = RateLimiter(max_per_minute=60)

print("✅ Rate limiter created")
print("💡 Use limiter.wait_if_needed() before each API call")

### 9.2 Error Handling

In [ ]:
def call_gemini_with_retry(prompt: str, max_retries: int = 3) -> Dict:
    """Call Gemini API with exponential backoff retry logic."""
    for attempt in range(max_retries):
        try:
            result = call_gemini(prompt)
            if 'error' not in result:
                return result
            else:
                raise Exception(result['error'])
        except Exception as e:
            if attempt == max_retries - 1:
                return {'response': f"Failed after {max_retries} attempts: {e}", 'error': str(e)}
            
            wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
            print(f"⚠️  Attempt {attempt + 1} failed: {e}")
            print(f"   Retrying in {wait_time}s...")
            time.sleep(wait_time)

print("✅ Robust error handling implemented")
print("💡 Always use retry logic with exponential backoff in production")

### 9.3 Caching Results

In [ ]:
import hashlib
from typing import Dict, Optional

class SimpleCache:
    """Simple in-memory cache for API responses."""
    
    def __init__(self):
        self.cache = {}
    
    def get_key(self, prompt: str) -> str:
        """Generate cache key from prompt."""
        return hashlib.md5(prompt.encode()).hexdigest()
    
    def get(self, prompt: str) -> Optional[Dict]:
        """Get cached response if exists."""
        key = self.get_key(prompt)
        return self.cache.get(key)
    
    def set(self, prompt: str, response: Dict):
        """Cache a response."""
        key = self.get_key(prompt)
        self.cache[key] = response

# Usage
cache = SimpleCache()

def call_gemini_cached(prompt: str) -> Dict:
    """Call Gemini with caching."""
    # Check cache first
    cached = cache.get(prompt)
    if cached:
        print("✨ Cache hit!")
        return cached
    
    # Not in cache, make API call
    result = call_gemini(prompt)
    cache.set(prompt, result)
    return result

print("✅ Caching implemented")
print("💡 Caching can save 50-80% of API costs for repeated queries")

---
## 10. Summary

### Key Takeaways

1. **Gemini is Perfect for Students**
   - ✅ Completely FREE (60 req/min, 1,500/day)
   - ✅ No credit card required
   - ✅ Good quality (comparable to GPT-3.5)
   - ✅ Easy to use

2. **Prompt Engineering is Critical**
   - Be specific and provide context
   - Use few-shot learning for consistency
   - Chain-of-thought for complex reasoning
   - Iterate and refine

3. **Evaluation is Multi-dimensional**
   - Accuracy (task-specific metrics)
   - Latency (user experience)
   - Cost (operational feasibility)
   - Quality (for subjective tasks)

4. **When to Use What**
   ```
   Learning/Prototyping → Gemini (FREE)
   Production (low-med volume) → Gemini Pro (paid)
   Production (critical quality) → GPT-4 / Claude 3.5
   Production (high volume) → Fine-tune smaller model
   Privacy-sensitive → Self-host LLaMA/Mistral
   ```

5. **Best Practices**
   - Implement rate limiting
   - Use retry logic with exponential backoff
   - Cache frequent queries
   - Monitor usage and performance
   - Never commit API keys to version control

### Next Steps

1. **Experiment**: Try different prompts and temperatures
2. **Build**: Create a simple chatbot or Q&A system
3. **Combine**: Use Gemini with RAG (see NLP14_2_1_RAG_with_Gemini.ipynb)
4. **Deploy**: Build a web interface with Streamlit
5. **Scale**: Learn about vector databases and production patterns

---

### Resources

- [Google AI Studio](https://makersuite.google.com/)
- [Gemini API Documentation](https://ai.google.dev/docs)
- [Prompt Engineering Guide](https://www.promptingguide.ai/)
- [Google Colab](https://colab.research.google.com/) - Free GPU notebooks

🎉 **You're now ready to build with modern LLMs using Gemini!**